LLM 파인튜닝(Fine-tuning)을 하기위해서는 LLM 고유의 챗 템플릿(Chat template)에 대해서 이해해야 합니다. 이번 실습에서는 챗 템플릿이 무엇이고 이를 어떻게 적용할 수 있는지에 대해서 알아봅시다.

## 1. 토크나이저

모든 LLM들은 고유한 토크나이저가 존재합니다. 우선 transformers 패키지로부터 AutoTokenizer를 임포트합니다.

In [ ]:
from transformers import AutoTokenizer

`AutoTokenizer.from_pretrained('모델 이름')`을 사용하여 LLM 고유의 토크나이저를 불러올 수 있습니다. 저희가 이번에 사용할 모델은 `Qwen/Qwen2-7B-Instruct"`입니다.

https://huggingface.co/Qwen/Qwen2-7B-Instruct

In [ ]:
model_id = "Qwen/Qwen2-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

토크나이저의 메소드 `encode()`는 사람의 입력을 LLM이 이해할 수 있는 정수로 변환하고 `decode()`는 다시 이를 사람의 입력으로 해석하게 해줍니다.

In [ ]:
# 토큰화 과정을 내부적으로 수행 후에 바로 인코딩 한 결과
# 토큰화 -> vocabulary 기반 인코딩
tokenizer.encode('안녕하세요. 반갑습니다.')

[126246, 144370, 91145, 13, 63757, 138685, 38231, 13]

위의 결과는 '안녕하세요. 반갑습니다.'를 토크나이저가 토큰화 후에 정수 인코딩을 수행한 후의 결과입니다.  
이를 다시 `decode()`의 입력으로 사용하여 복원해봅시다.

In [ ]:
tokenizer.decode([126246, 144370, 91145, 13, 63757, 138685, 38231, 13])

'안녕하세요. 반갑습니다.'

다시 '안녕하세요. 반갑습니다.' 문자열로 복원된 것을 볼 수 있습니다.

## 템플릿

https://huggingface.co/docs/transformers/main/en/chat_templating

템플릿은 해당 모델이 만들어졌을 당시의 모델의 입력으로 넣을 때 반드시 지켜주어야 할 입력 형식입니다. 이는 모델마다 다르며 이미 학습된 LLM을 사용할 때 해당 LLM의 챗 템플릿을 따르지 않으면 제대로 된 성능을 얻기 어렵습니다. 모델 사용 전 반드시 기억해야 합니다. 이번 실습에서는 `"Qwen/Qwen2-7B-Instruct"` 모델의 챗 템플릿에 대해서 알아볼 것입니다.

해당 모델의 챗 템플릿을 알아보기 전에 우리가 해당 모델의 입력으로 넣을 시스템 프롬프트와 유저 프롬프트를 오픈 AI 형식으로 작성해보겠습니다. 오픈 AI 형식이란, 오픈 AI의 GPT-4 API를 사용할 때의 형식을 의미합니다. 앞서 GPT-4 API를 사용하며 시스템 프롬프트와 유저 프롬프트에 대해서 설명한 바 있습니다.

- 시스템 프롬프트: 모델의 역할이나 모델이 앞으로 해야할 일을 적는 구간.
- 유저 프롬프트: 모델에게 지금 요청할 것 (사용자의 질문)

임의로 아래와 같이 시스템 프롬프트와 유저 프롬프트를 GPT-4 API를 사용할 당시의 형식으로 작성해보았습니다.

```
messages = [
    {"role": "system", "content": "우리가 챗봇에게 바라는 역할"},
    {"role": "user", "content": "사용자가 실제로 넣는 입력"}
]
```

In [ ]:
messages = [
    {"role": "system", "content": "당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요."},
    {"role": "user", "content": "은행의 기준 금리에 대해서 설명해줘"}
]

모델에게 입력할 시스템 프롬프트와 유저 프롬프트를 오픈 AI 형식으로 작성하였고, 사용할 모델의 토크나이저가 로드된 상황이라면 챗 템플릿 형식으로 빠르게 적용할 수 있습니다. `tokenizer.apply_chat_template()`을 사용하되 입력으로 오픈 AI 형식으로 작성된 프롬프트를 넣으면 해당 LLM이 사용하고 있는 챗 템플릿으로 자동 변환해줍니다.

tokenize=False를 사용하면 템플릿 적용 후의 결과를 확인할 수 있습니다.

In [ ]:
# tokenizer=False는 인코딩은 안 하고 챗 템플릿만 적용
template_messages = tokenizer.apply_chat_template(messages, tokenize=False)
print(template_messages)

<|im_start|>system
당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요.<|im_end|>
<|im_start|>user
은행의 기준 금리에 대해서 설명해줘<|im_end|>



위의 결과를 보면 시스템 프롬프트는 `<|im_start|>`와 `<|im_end|>` 사이에 작성되고, 유저 프롬프트 또한 `<|im_start|>`와 `<|im_end|>` 사이에 작성되어져 있는데 해당 형태가 `"Qwen/Qwen2-7B-Instruct"` 모델의 챗 템플릿입니다.

해당 모델을 사용할 때는 항상 시스템 프롬프트와 유저 프롬프트가 위의 형태를 가지도록 하여 모델의 입력으로 넣는 것이 좋습니다. 물론 모델에 넣을 때는 위 형식에서 정수 인코딩까지 되어져 있어야만 합니다. `tokenizer=True`를 사용하면 템플릿 적용 및 정수 인코딩 후의 결과를 볼 수 있습니다. 하지만 `tokenizer=True`가 기본 값이므로 `tokenizer` 인자의 값을 기재하지 않으면 됩니다.

tokenizer=True를 사용하면 템플릿 적용 및 정수 인코딩 후의 결과를 볼 수 있습니다.

In [ ]:
encodeds = tokenizer.apply_chat_template(messages)
print('템플릿 적용 및 정수 인코딩 후:\n', encodeds)

템플릿 적용 및 정수 인코딩 후:
 {'input_ids': [151644, 8948, 198, 64795, 82528, 33704, 58677, 78125, 21329, 66019, 124685, 134619, 94152, 28626, 78952, 13, 133552, 119, 16560, 126254, 19391, 90711, 250, 126550, 126204, 36055, 133085, 128555, 143604, 91145, 13, 151645, 198, 151644, 872, 198, 33704, 124528, 20401, 54116, 129044, 40771, 230, 28002, 19391, 130869, 133828, 33883, 144131, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


실제로 위 결과를 다시 decode()하면 이미 챗 템플릿이 적용된 상태임을 확인할 수 있습니다.

In [ ]:
print('--' * 100)
print('템플릿 적용 및 정수 인코딩 결과를 복원:\n',tokenizer.decode((encodeds['input_ids'])))

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
템플릿 적용 및 정수 인코딩 결과를 복원:
 <|im_start|>system
당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요.<|im_end|>
<|im_start|>user
은행의 기준 금리에 대해서 설명해줘<|im_end|>



## 3. 생성 프롬프트(Generation Prompt)

챗 템플릿을 적용할 때 추가로 고려해야 할 것으로 생성 프롬프트(Generation Prompt)라는 것이 있습니다. LLM이 적절하게 응답을 할 수 있도록 도와주는 프롬프트입니다. `apply_chat_template()`을 사용할 때 `add_generation_prompt=True`를 사용하면 해당 프롬프트를 확인할 수 있습니다.

In [ ]:
encodeds = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
print(encodeds)

{'input_ids': [151644, 8948, 198, 64795, 82528, 33704, 58677, 78125, 21329, 66019, 124685, 134619, 94152, 28626, 78952, 13, 133552, 119, 16560, 126254, 19391, 90711, 250, 126550, 126204, 36055, 133085, 128555, 143604, 91145, 13, 151645, 198, 151644, 872, 198, 33704, 124528, 20401, 54116, 129044, 40771, 230, 28002, 19391, 130869, 133828, 33883, 144131, 151645, 198, 151644, 77091, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


정수 인코딩이 된 결과가 나와서 `add_generation_prompt=True`를 추가하므로서 현재 챗 템플릿에 어떤 변화가 일어났는지 위 결과만으로는 확인이 어렵습니다. 위 결과를 `decode()`를 통해 정수 시퀀스를 텍스트 형태로 변환하여 챗 템플릿에 어떤 변화가 있었는지 보다 쉽게 확인해보겠습니다.

In [ ]:
print('템플릿 적용 및 정수 인코딩 결과를 복원:\n',tokenizer.decode((encodeds['input_ids'])))

템플릿 적용 및 정수 인코딩 결과를 복원:
 <|im_start|>system
당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요.<|im_end|>
<|im_start|>user
은행의 기준 금리에 대해서 설명해줘<|im_end|>
<|im_start|>assistant



이전의 챗 템플릿과는 달리 `<|im_start|>assistant`가 추가된 것을 확인할 수 있습니다. 이것이 생성 프롬프트입니다. 이 프롬프트가 어떤 의미를 가지는지 좀 더 쉽게 이해하기 위해서 다양한 예시의 챗 템플릿으로 이해해보겠습니다. 예를 들어서 어떤 LLM이 아래와 같은 챗 템플릿 형태로 학습했다고 가정해봅시다.

## 4. 예시로 알아보는 챗 템플릿의 이해

LLM이 학습되었을 당시의 챗 템플릿  
```
system: 너는 사용자의 질문에 친절하게 답변해야해. end:
user: 안녕 end:
assistant: 반갑습니다. end:
```

위 형식의 챗 템플릿으로 학습된 LLM 사용 시, LLM의 입력으로 사용하고 싶은 시스템 프롬프트와 유저 프롬프트가 있다면 아래와 같이 학습된 챗 템플릿 형식에 맞춰 배치해야 합니다. 예를 들어서 시스템 프롬프트로 '당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요.'를 사용하고 유저 프롬프트로 '은행의 기준 금리에 대해서 설명해줘'를 사용하여 LLM의 답변을 얻고자 한다면 아래와 같이 입력을 넣어야 합니다.

사용할 때
```
system: 당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요. end:
user: 은행의 기준 금리에 대해서 설명해줘 end:
assistant:
```

LLM은 학습 당시에 `assistant:` 다음부터 실제 답변을 작성하도록 학습이 되어져 있기 때문에 LLM에 위와 같은 입력을 넣어주어야 하는 것입니다. 위의 입력을 넣은 LLM은 아래와 같이 답변을 생성할 것입니다. 입력의 챗 템플릿의 끝에 이미 `assistant:`가 붙어있기 때문에 이어서 답변만 작성하면 됩니다.

```
중앙은행이 정하는 기준이 되는 금리로, 시중 금리와 경제 전반에 영향을 미칩니다. :end
```

또 다른 예를 들어봅시다. 예를 들어서 어떤 LLM이 아래와 같은 챗 템플릿 형태로 학습했다고 가정해봅시다.
```python
<system> 너는 사용자의 질문에 친절하게 답변해야해. </system>  
<user> 안녕 </user>  
<assistant> 반갑습니다. </assistant>
```
위 형식의 챗 템플릿으로 학습된 LLM을 사용한다고 가정하였을 때, 시스템 프롬프트와 유저 프롬프트를 새로 입력한다면 아래와 같이 학습된 챗 템플릿 형식에 맞춰 배치해야 합니다.
```python
<system> 당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요. </system>   
<user> 은행의 기준 금리에 대해서 설명해줘 </user>   
<assistant>
```
LLM은 학습 당시에 `<assistant>` 다음부터 실제 답변을 작성하도록 학습이 되어져 있기 때문에 위와 같이 `<assistant>`까지 부착하여 입력을 넣어주어야 하는 것입니다. 위의 입력을 넣은 LLM은 아래와 같이 답변을 생성할 것입니다. 입력으로 사용된 챗 템플릿의 끝에 이미 `<assistant>`가 붙어있기 때문에 바로 답변만 작성하면 됩니다.
```python
중앙은행이 정하는 기준이 되는 금리로, 시중 금리와 경제 전반에 영향을 미칩니다. </assistant>
```
방금 설명한 두 가지 예시에서는 생성 프롬프트가 각각 `assistant:`와 `<assistant>`에 해당됩니다.  


다시 말해 생성 프롬프트란 LLM이 훈련된 챗 템플릿 형식에서 응답을 시작해야 하는 위치를 표시하는 부분입니다. `apply_chat_template()`을 사용할 때  `add_generation_prompt=True` 옵션을 사용하면 각 모델에 맞는 생성 프롬프트가 자동으로 추가되어 LLM이 답변 시 학습 시의 성능을 보존한 채 올바른 응답 생성이 가능해집니다.  

지금까지 LLM에게 사용자의 입력을 전달할 때 반드시 알아야 할 개념인 챗 템플릿과 생성 프롬프트에 대해서 정리해보았습니다. GPT-4 API와 같은 API 서비스에서는 오픈AI 형식과 같이 특정 형식으로 전달하면 내부적으로 챗 템플릿이 적용되어 LLM이 호출되므로 API 사용자가 크게 신경 쓸 필요가 없지만, 모델을 다운받아서 직접 사용하는 경우에는 이러한 챗 템플릿을 적절히 적용하여 입력을 전달해야 합니다.  

챗 템플릿이나 생성 프롬프트를 올바르게 적용하지 않으면 다운로드하여 사용 중인 모델의 답변 성능이 크게 저하될 수 있습니다. 또는 파인 튜닝을 한다고 하더라도 챗 템플릿을 무시하고 파인 튜닝을 한 모델은 본래 LLM의 성능을 제대로 사용할 수 없습니다. 따라서 실제 운영 환경에서 모델을 다운받아서 활용할 때는 반드시 해당 모델의 챗 템플릿과 생성 프롬프트를 정확히 적용해야 할 것입니다.

## Gemma4 템플릿 확인하기

모델마다 템플릿이 완전히 다릅니다. 이번에는 Qwen2가 아닌 Gemma4라는 또 다른 모델의 템플릿을 확인해봅시다.

In [ ]:
model_id = "google/gemma-4-E4B-it"
gemma_tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

In [ ]:
messages = [
    {"role": "system", "content": "당신은 러닝스푼즈 챗봇 스푼이입니다. 묻는 말에 친절하고 정확하게 답변하세요."},
    {"role": "user", "content": "당신이 누군지 설명해주십시오"},
    {"role": "assistant", "content": "저는 챗봇 스푼이랍니다."},
    {"role": "user", "content": "어디 소속인데?"},
]

encodeds = gemma_tokenizer.apply_chat_template(messages, add_generation_prompt=True)
print(encodeds)

{'input_ids': [2, 105, 9731, 107, 238749, 238144, 237456, 139946, 242358, 237553, 246741, 239611, 236743, 246214, 243749, 18282, 246741, 237077, 15245, 236761, 236743, 244209, 237170, 18906, 237223, 63570, 239498, 14377, 143474, 26216, 231757, 152748, 236761, 106, 107, 105, 2364, 107, 238749, 238144, 237077, 50030, 239501, 237308, 59587, 113530, 111462, 106, 107, 105, 4368, 107, 234883, 236743, 246214, 243749, 18282, 246741, 237077, 125634, 236761, 106, 107, 105, 2364, 107, 237430, 238945, 18004, 238701, 58123, 236881, 106, 107, 105, 4368, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
print(gemma_tokenizer.decode((encodeds['input_ids'])))

<bos><|turn>system
당신은 러닝스푼즈 챗봇 스푼이입니다. 묻는 말에 친절하고 정확하게 답변하세요.<turn|>
<|turn>user
당신이 누군지 설명해주십시오<turn|>
<|turn>model
저는 챗봇 스푼이랍니다.<turn|>
<|turn>user
어디 소속인데?<turn|>
<|turn>model



멀티턴 상황

In [ ]:
messages = [
    {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
    {"role": "user", "content": "안녕하세요, 오늘 날씨는 어떤가요?"},
    {"role": "assistant", "content": "안녕하세요! 오늘 날씨는 맑고 화창합니다."},
    {"role": "user", "content": "오 그렇다면 놀러가기 좋겠어요!"},
]

encodeds = gemma_tokenizer.apply_chat_template(messages, add_generation_prompt=True)
print(encodeds)

{'input_ids': [2, 105, 9731, 107, 238749, 238144, 237456, 63570, 239498, 237384, 12498, 10443, 237462, 237553, 241806, 237947, 15245, 236761, 106, 107, 105, 2364, 107, 61659, 236764, 31694, 57517, 240482, 237170, 51955, 237272, 237586, 236881, 106, 107, 105, 4368, 107, 61659, 236888, 31694, 57517, 240482, 237170, 236743, 245528, 237296, 34987, 240023, 19773, 236761, 106, 107, 105, 2364, 107, 237959, 57636, 98997, 105166, 238182, 237272, 237351, 27694, 238810, 31990, 236888, 106, 107, 105, 4368, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
print(gemma_tokenizer.decode((encodeds['input_ids'])))

<bos><|turn>system
당신은 친절한 AI 어시스턴트입니다.<turn|>
<|turn>user
안녕하세요, 오늘 날씨는 어떤가요?<turn|>
<|turn>model
안녕하세요! 오늘 날씨는 맑고 화창합니다.<turn|>
<|turn>user
오 그렇다면 놀러가기 좋겠어요!<turn|>
<|turn>model



In [ ]:
print(encodeds['input_ids'])

[2, 105, 9731, 107, 238749, 238144, 237456, 63570, 239498, 237384, 12498, 10443, 237462, 237553, 241806, 237947, 15245, 236761, 106, 107, 105, 2364, 107, 61659, 236764, 31694, 57517, 240482, 237170, 51955, 237272, 237586, 236881, 106, 107, 105, 4368, 107, 61659, 236888, 31694, 57517, 240482, 237170, 236743, 245528, 237296, 34987, 240023, 19773, 236761, 106, 107, 105, 2364, 107, 237959, 57636, 98997, 105166, 238182, 237272, 237351, 27694, 238810, 31990, 236888, 106, 107, 105, 4368, 107]


## Qwen3 템플릿

Qwen3부터는 답변 시 `<think>생각</think>`를 이용하여 생각하고 답변합니다.

In [ ]:
model_id = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
messages = [
    {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
    {"role": "user", "content": "사과 3개에 2개를 더 사면 총 몇 개야?"},
]

encodeds = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
print(encodeds)

{'input_ids': [151644, 8948, 198, 64795, 82528, 33704, 90711, 250, 126550, 23573, 15235, 124685, 134619, 94152, 28626, 78952, 13, 151645, 198, 151644, 872, 198, 55054, 53680, 220, 18, 59761, 19391, 220, 17, 59761, 18411, 126366, 32129, 32290, 3315, 112, 251, 36978, 229, 73523, 89659, 30, 151645, 198, 151644, 77091, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
print(tokenizer.decode((encodeds['input_ids'])))

<|im_start|>system
당신은 친절한 AI 어시스턴트입니다.<|im_end|>
<|im_start|>user
사과 3개에 2개를 더 사면 총 몇 개야?<|im_end|>
<|im_start|>assistant



예를 들어서 위와 같은 입력을 구성하고 Qwen3에 넣으면 Qwen3은 기본적으로 아래와 같이 답변합니다.

```
<think>
사용자가 사과 개수를 묻고 있다. 처음에 3개가 있고 2개를 더 산다.
3 + 2 = 5. 단순 덧셈이다.
</think>

총 5개입니다.<|im_end|>
```

여기서 <think>와 </think> 사이에 들어간 내용은 모델이 답을 내기 전에 거치는 **사고 과정(reasoning)**입니다. 사람으로 치면 머릿속으로 "음, 3개에 2개니까 3+2겠지" 하고 중얼거리는 부분이라고 보시면 됩니다. 이것은 최종 답변이 아니라 답변을 만들기 위한 작업 공간이기 때문에, 보통 사용자에게 그대로 보여주지 않습니다. </think> 다음 빈 줄이 나오고, 그 뒤에 오는 총 5개입니다.가 실제로 사용자에게 전달할 정답입니다.  

그래서 실제 정답만 뽑아내려면 파싱을 해주어야 합니다. 모델 출력 문자열에서 <think>로 시작해 </think>로 끝나는 블록을 잘라내고, 그 뒤에 남는 텍스트만 취하면 됩니다. 예를 들면 다음과 같습니다.

In [ ]:
import re

output = """<think>
사용자가 사과 개수를 묻고 있다. 처음에 3개가 있고 2개를 더 산다.
3 + 2 = 5. 단순 덧셈이다.
</think>

총 5개입니다."""

# think 블록 통째로 제거
answer = re.sub(r"<think>.*?</think>", "", output, flags=re.DOTALL).strip()

print('실제 응답 파싱')
print('==' * 50)
print(answer)   # 총 5개입니다.

실제 응답 파싱
총 5개입니다.


think를 받고 싶지 않을 때는 챗 템플릿이 `<think></think>` 사이를 빈 블록으로 값을 채워서 입력으로 보내버리는 식으로 사용합니다. 즉 모델이 생각을 시작할 자리를 템플릿이 먼저 막아버리는 방식입니다.

In [ ]:
messages = [
    {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
    {"role": "user", "content": "사과 3개에 2개를 더 사면 총 몇 개야?"},
]

encodeds = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False)

print(tokenizer.decode((encodeds['input_ids'])))

<|im_start|>system
당신은 친절한 AI 어시스턴트입니다.<|im_end|>
<|im_start|>user
사과 3개에 2개를 더 사면 총 몇 개야?<|im_end|>
<|im_start|>assistant
<think>

</think>




예를 들어서 위와 같은 입력을 구성하고 Qwen3에 넣으면, 모델 입장에서는 이미 <think></think> 블록이 닫힌 채로 자기 차례가 시작된 상태입니다. 생각을 적을 자리가 템플릿 단계에서 막혀 있는 셈이라, 모델은 추론 과정을 건너뛰고 곧바로 최종 답변만 이어서 생성합니다.

```
총 5개입니다.<|im_end|>
```